# MG Score Distributions by Text Length

Compare stored classifier `mg_score` values for human-generated (HG) and machine-generated (MG) texts across token-length bins and feature sets.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from IPython.display import display

from scripts.experiment_config import FEATURE_SET_LABELS

from notebooks.utils.mg_score_length_distributions_utils import (
    load_bounded_prefix_result_frames,
    load_mg_score_length_frame,
    plot_bounded_prefix_macro_f1_comparison,
    plot_bounded_prefix_evaluation,
    dataset_balanced_bounded_prefix_feature_set_metrics,
    pooled_bounded_prefix_feature_set_metrics,
    plot_mg_score_by_length,
)
from notebooks.utils.report_diagrams_utils import save_png_figure

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PLOT_OUTPUT_ROOT = DATA_ROOT / "plots" / "mg_score_length_distributions"
PLOT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Parameters

Each plot uses one evaluation dataset and classifier. Feature sets are shown as separate panels with shared HG/MG colors and score scale.

In [ ]:
RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
SCORER_LLM = "gemma_3_270m_it"

FEATURE_SETS = ("simple_similarity", "scalar_all")
MODEL_FAMILIES = ("logistic_regression", "random_forest")
TRAINING_COMPOSITION = "Kaggle+MAGE+Outfox+BBC"
EVALUATION_COMPOSITIONS = ("kaggle_train", "mage_train", "outfox_chatgpt_train")
PLOT_DATASETS = (*EVALUATION_COMPOSITIONS, "mixed")
FEATURE_SET_DISPLAY_NAMES = {**FEATURE_SET_LABELS, "scalar_all": "probabilistic"}
TRAINING_WINDOW = 32
LENGTH_BIN_SIZE = 100

BOUNDED_PREFIX_RUN_ID = "bounded-prefix-classifier-evaluation-gemma270m-rf-window32-2026-09-29"
BOUNDED_PREFIX_MODEL_FAMILY = "random_forest"
BOUNDED_PREFIX_SCORER_LLM = "gemma_3_270m_it"
BOUNDED_PREFIX_FEATURE_SET = "simple_similarity"
BOUNDED_PREFIX_TRAINING_REGIME = "Kaggle+MAGE+Outfox+BBC"
BOUNDED_PREFIX_TRAINING_WINDOW = 32
BOUNDED_PREFIX_EVALUATION_REGIME = "Mixed"
BOUNDED_PREFIX_PARTITION = "validation"

## Load Selected Predictions

In [ ]:
score_frame = load_mg_score_length_frame(
    DATA_ROOT,
    run_id=RUN_ID,
    scorer_llm=SCORER_LLM,
    feature_sets=FEATURE_SETS,
    training_window=TRAINING_WINDOW,
    model_families=MODEL_FAMILIES,
    training_composition=TRAINING_COMPOSITION,
    evaluation_compositions=EVALUATION_COMPOSITIONS,
)
print(f"Rows: {len(score_frame):,}")
display(score_frame.groupby(["feature_set", "dataset", "model_family", "label"], observed=True).size().rename("rows").reset_index())

## HG/MG Score Distributions

Each figure is one evaluation dataset and one classifier. Each feature set has its own panel, while HG and MG distributions use the same colors and y-axis scale.

In [ ]:
if "score_frame" not in globals():
    score_frame = load_mg_score_length_frame(
        DATA_ROOT,
        run_id=RUN_ID,
        scorer_llm=SCORER_LLM,
        feature_sets=FEATURE_SETS,
        training_window=TRAINING_WINDOW,
        model_families=MODEL_FAMILIES,
        training_composition=TRAINING_COMPOSITION,
        evaluation_compositions=EVALUATION_COMPOSITIONS,
    )

plot_frame = pd.concat(
    [score_frame, score_frame.assign(dataset="mixed")],
    ignore_index=True,
)

for dataset in PLOT_DATASETS:
    for model_family in MODEL_FAMILIES:
        figure = plot_mg_score_by_length(
            plot_frame,
            dataset=dataset,
            model_family=model_family,
            length_bin_size=LENGTH_BIN_SIZE,
        )
        for axis, feature_set in zip(figure.axes, FEATURE_SETS):
            axis.set_title(FEATURE_SET_DISPLAY_NAMES.get(feature_set, feature_set))
        save_png_figure(figure, PLOT_OUTPUT_ROOT / f"mg_score_by_length__{dataset}__{model_family}__{RUN_ID}.png")
        display(figure)

In [ ]:
# Presentation-ready summary for the mixed evaluation data.
import matplotlib.pyplot as plt
presentation_frame = plot_frame.loc[
    plot_frame["dataset"].eq("mixed")
    & plot_frame["feature_set"].isin(("scalar_all", "simple_similarity"))
    & plot_frame["token_count"].between(100, 1199)
].copy()
presentation_frame["length_bin"] = (presentation_frame["token_count"] // 100).astype(int) * 100
presentation_frame["length_label"] = presentation_frame["length_bin"].map(lambda value: f"{value}-{value + 99}")
length_order = [f"{value}-{value + 99}" for value in range(100, 1200, 100)]
feature_specs = tuple((feature_set, FEATURE_SET_DISPLAY_NAMES.get(feature_set, feature_set), cutoff) for feature_set, cutoff in (("scalar_all", 600), ("simple_similarity", 300)))
colors = {"HG": "#2F6690", "MG": "#D1495B"}
for model_family in MODEL_FAMILIES:
    model_frame = presentation_frame.loc[presentation_frame["model_family"].eq(model_family)]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.8), sharey=True)
    for axis, (feature_set, title, cutoff) in zip(axes, feature_specs):
        feature_frame = model_frame.loc[model_frame["feature_set"].eq(feature_set)]
        sns.boxplot(data=feature_frame, x="length_label", y="mg_score", hue="label",
                    hue_order=("HG", "MG"), order=length_order, palette=colors,
                    width=0.72, linewidth=1.0, fliersize=0, ax=axis)
        cutoff_position = (cutoff - 100) // 100 - 0.5
        axis.axvspan(cutoff_position, len(length_order) - 0.5, color="#F2C14E", alpha=0.13, zorder=0)
        axis.axvline(cutoff_position, color="#B7791F", linewidth=2.2, linestyle="--", zorder=4)
        axis.text(cutoff_position + 0.12, 0.96, f"Recommended minimum\n{cutoff} tokens",
                  transform=axis.get_xaxis_transform(), ha="left", va="top",
                  fontsize=12, fontweight="bold", color="#7A4E00",
                  bbox={"boxstyle": "round,pad=0.35", "facecolor": "#FFF7D6", "edgecolor": "#E3B341"})
        axis.set_title(title, fontsize=18, fontweight="bold", pad=14)
        axis.set_xlabel("Text length (tokens)", fontsize=12, labelpad=10)
        axis.set_ylabel("MG score" if axis is axes[0] else "", fontsize=12)
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=45, labelsize=9)
        axis.grid(axis="y", color="#D9E2EC", linewidth=0.8)
        axis.grid(axis="x", visible=False)
        axis.set_axisbelow(True)
        axis.spines[["top", "right"]].set_visible(False)
        if axis is axes[1]:
            axis.get_legend().remove()
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles, labels, title="Class", frameon=False, loc="lower left")
    fig.suptitle(f"{model_family.replace('_', ' ').title()}: more text makes the signal easier to separate", fontsize=23, fontweight="bold", y=1.02)
    fig.text(0.5, 0.965, "Mixed evaluation data | shaded area marks the usable range", ha="center", fontsize=12, color="#52606D")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    save_png_figure(fig, PLOT_OUTPUT_ROOT / f"presentation_mg_score_length__{model_family}__{RUN_ID}.png")
    display(fig)

In [ ]:
# Compact-label companion for presentation slides.
major_tick_positions = tuple(range(0, len(length_order), 2))
major_tick_labels = tuple(f"{100 + position * 100}" for position in major_tick_positions)

for model_family in MODEL_FAMILIES:
    model_frame = presentation_frame.loc[presentation_frame["model_family"].eq(model_family)]
    fig, axes = plt.subplots(1, 2, figsize=(16, 7.1), sharey=True)
    for axis, (feature_set, title, cutoff) in zip(axes, feature_specs):
        feature_frame = model_frame.loc[model_frame["feature_set"].eq(feature_set)]
        sns.boxplot(data=feature_frame, x="length_label", y="mg_score", hue="label",
                    hue_order=("HG", "MG"), order=length_order, palette=colors,
                    width=0.72, linewidth=1.0, fliersize=0, ax=axis)
        cutoff_position = (cutoff - 100) // 100 - 0.5
        axis.axvspan(cutoff_position, len(length_order) - 0.5, color="#F2C14E", alpha=0.13, zorder=0)
        axis.axvline(cutoff_position, color="#B7791F", linewidth=2.2, linestyle="--", zorder=4)
        axis.text(cutoff_position + 0.12, 0.96, f"Recommended minimum\n{cutoff} tokens",
                  transform=axis.get_xaxis_transform(), ha="left", va="top",
                  fontsize=12, fontweight="bold", color="#7A4E00",
                  bbox={"boxstyle": "round,pad=0.35", "facecolor": "#FFF7D6", "edgecolor": "#E3B341"})
        axis.set_title(title, fontsize=18, fontweight="bold", pad=14)
        axis.set_xlabel("Text length (tokens, 100-token bins)", fontsize=12, labelpad=10)
        axis.set_ylabel("MG score" if axis is axes[0] else "", fontsize=12)
        axis.set_ylim(0, 1)
        axis.set_xticks(major_tick_positions, major_tick_labels)
        axis.tick_params(axis="x", rotation=0, labelsize=10)
        axis.grid(axis="y", color="#D9E2EC", linewidth=0.8)
        axis.grid(axis="x", visible=False)
        axis.set_axisbelow(True)
        axis.spines[["top", "right"]].set_visible(False)
        if axis is axes[1]:
            axis.get_legend().remove()
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles, labels, title="Class", frameon=False, loc="lower left")
    fig.suptitle(f"{model_family.replace('_', ' ').title()}: more text makes the signal easier to separate",
                 fontsize=23, fontweight="bold", y=1.045)
    fig.text(0.5, 0.935, "Mixed evaluation data | shaded area marks the usable range",
             ha="center", fontsize=12, color="#52606D")
    fig.tight_layout(rect=(0, 0, 1, 0.89))
    save_png_figure(fig, PLOT_OUTPUT_ROOT / f"presentation_mg_score_length_compact__{model_family}__{RUN_ID}.png")
    display(fig)


## Bounded-Prefix Evaluation

The left panel pools dataset confusion counts to show balanced accuracy and macro F1 by leading-prefix length. ROC-AUC is omitted because it cannot be reconstructed from the per-dataset metric export. The right panel shows HG/MG scored-text coverage for each source dataset.

In [ ]:
prefix_metrics, prefix_coverage = load_bounded_prefix_result_frames(
    DATA_ROOT,
    run_id=BOUNDED_PREFIX_RUN_ID,
    model_family=BOUNDED_PREFIX_MODEL_FAMILY,
    scorer_llm=BOUNDED_PREFIX_SCORER_LLM,
    feature_set=BOUNDED_PREFIX_FEATURE_SET,
    training_regime=BOUNDED_PREFIX_TRAINING_REGIME,
    training_window=BOUNDED_PREFIX_TRAINING_WINDOW,
    evaluation_regime=BOUNDED_PREFIX_EVALUATION_REGIME,
    partition=BOUNDED_PREFIX_PARTITION,
)
prefix_figure = plot_bounded_prefix_evaluation(prefix_metrics, prefix_coverage)
save_png_figure(prefix_figure, PLOT_OUTPUT_ROOT / f"bounded_prefix_evaluation__{BOUNDED_PREFIX_RUN_ID}__{BOUNDED_PREFIX_PARTITION}.png")
display(prefix_figure)

In [ ]:
comparison_metrics_by_feature_set = {}
for feature_set in FEATURE_SETS:
    metrics, _ = load_bounded_prefix_result_frames(
        DATA_ROOT,
        run_id=BOUNDED_PREFIX_RUN_ID,
        model_family=BOUNDED_PREFIX_MODEL_FAMILY,
        scorer_llm=BOUNDED_PREFIX_SCORER_LLM,
        feature_set=feature_set,
        training_regime=BOUNDED_PREFIX_TRAINING_REGIME,
        training_window=BOUNDED_PREFIX_TRAINING_WINDOW,
        evaluation_regime=BOUNDED_PREFIX_EVALUATION_REGIME,
        partition=BOUNDED_PREFIX_PARTITION,
    )
    comparison_metrics_by_feature_set[feature_set] = metrics

comparison_metrics = dataset_balanced_bounded_prefix_feature_set_metrics(comparison_metrics_by_feature_set)
comparison_figure = plot_bounded_prefix_macro_f1_comparison(
        comparison_metrics,
        feature_set_order=FEATURE_SETS,
        bound_labels=("100-300", "300-600", "600-1200"),
    )
save_png_figure(comparison_figure, PLOT_OUTPUT_ROOT / f"bounded_prefix_feature_set_comparison__{BOUNDED_PREFIX_RUN_ID}__{BOUNDED_PREFIX_PARTITION}.png")
display(comparison_figure)

In [ ]:
pooled_comparison_metrics = pooled_bounded_prefix_feature_set_metrics(
    comparison_metrics_by_feature_set,
)
pooled_comparison_metrics = pooled_comparison_metrics.assign(
    token_bound=pooled_comparison_metrics.apply(
        lambda row: f"{int(row['min_tokens'])}-{int(row['max_tokens'])}",
        axis=1,
    )
)

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
sns.barplot(
    data=pooled_comparison_metrics,
    x="token_bound",
    y="macro_f1",
    hue="feature_set",
    order=["100-300", "300-600", "600-1200"],
    hue_order=FEATURE_SETS,
    errorbar=None,
    ax=axis,
)
for container in axis.containers:
    axis.bar_label(container, fmt="%.3f", padding=3)

axis.set_title("Pooled bounded-prefix macro F1")
axis.set_xlabel("Leading prefix length (tokens)")
axis.set_ylabel("Macro F1")
axis.set_ylim(0.6, 1)
axis.legend(title="Feature set")
save_png_figure(figure, PLOT_OUTPUT_ROOT / f"bounded_prefix_pooled_comparison__{BOUNDED_PREFIX_RUN_ID}__{BOUNDED_PREFIX_PARTITION}.png")
display(figure)

### Pooled versus dataset-balanced bounded-prefix macro F1

This diagram pools the confusion counts across datasets before calculating macro F1. Consequently, datasets with more evaluated texts contribute more to the result. The previous diagram calculates macro F1 separately for each dataset and then averages those values equally, so every dataset contributes the same weight. Differences between the two diagrams reflect dataset weighting, not a change in the model, scorer, feature computation, or prefix ranges.